## Bonus Exercise - Fast Fractal Fun with Cython & GPUs

## Task B.1.

We set up our Cython the following way:

In [1]:
# %load setup.py
from setuptools import Extension, setup
from Cython.Build import cythonize
import numpy

extensions = [
    Extension(
        "*", ["*.pyx"],
        include_dirs=[numpy.get_include()]
    ),
]

setup(
    ext_modules=cythonize(
        extensions,
        compiler_directives={"language_level": "3"}
    )
)


Then we started converting the code to Cython, we disabled the wraparound and boundary checks, assigned type to variables, arguments and functions, then converted the Numpy arrays to memoryviews, changed slighty the complex number assigment to `c`, and lastly converted `mandelbrot_set` to a cpython function to reduce python calls, and `mandelbrot` function to a pure inline C function and disabled Python exception checks, since it won't be called from a python function. This made it so that in the code only numpy makes Python calls, according to Cython's visualization tool.

In [2]:
# %load cmandelbrot.pyx
cimport cython
import numpy as np
# noinspection PyUnresolvedReferences
cimport numpy as np

np.import_array()

@cython.boundscheck(False)
@cython.wraparound(False)
cpdef double[:, :] mandelbrot_set(unsigned int width, unsigned int height, double x_min, double x_max, double y_min, double y_max, unsigned int max_iter=100):
    """Generates the Mandelbrot set image."""
    cdef double[:] x_vals = np.linspace(x_min, x_max, width, dtype=np.double)
    cdef double[:] y_vals = np.linspace(y_min, y_max, height, dtype=np.double)
    cdef double[:, :] image = np.empty((height, width), dtype=np.double)
    cdef unsigned int i, j
    cdef double complex c

    for i in range(height):
        for j in range(width):
            c = x_vals[j] + y_vals[i] * 1j
            image[i, j] = mandelbrot(c, max_iter)

    return image

cdef inline unsigned int mandelbrot(double complex c, unsigned int max_iter=100) noexcept:
    """Computes the number of iterations before divergence."""
    cdef double complex z = 0
    cdef unsigned int n
    for n in range(max_iter):
        if abs(z) > 2:
            return n
        z = z * z + c
    return max_iter

## Task B.2

The code was first vectorized using Numpy and tested by comparing the output images, then the code was ported to PyTorch almost line by line with minimal changes.

We got rid of all the loops and are using 2 masks to keep track of which cells have diverged. The `mask` keeps track of all the old divergences so we don't change them. The `diverged` keeps track of cells that have diverged during current iteration.

In [1]:
# %load tmandelbrot.py
import torch as th
import matplotlib.pyplot as plt

def tmandelbrot(C, max_iter=100):
    """Computes the number of iterations before divergence."""
    device = C.device

    image = th.full(C.shape, max_iter, dtype=th.int32, device=device)
    Z = th.zeros(C.shape, dtype=th.complex128, device=device)
    mask = th.ones(C.shape, dtype=th.bool, device=device)
    diverged = th.zeros(C.shape, dtype=th.bool, device=device)

    for n in range(max_iter):
        diverged[mask] = th.abs(Z[mask]) > 2
        image[diverged] = n
        mask[diverged] = False
        diverged.fill_(False)

        Z[mask] = Z[mask] * Z[mask] + C[mask]

        if not mask.any(): break
    return image


def tmandelbrot_set(width, height, x_min, x_max, y_min, y_max, max_iter=100):
    """Generates the Mandelbrot set image."""
    device = th.device("cuda")
    x_vals = th.linspace(x_min, x_max, width, dtype=th.float64, device=device)
    y_vals = th.linspace(y_min, y_max, height, dtype=th.float64, device=device)

    real, imag = th.meshgrid(x_vals, y_vals, indexing="xy")
    C = th.complex(real, imag)

    return tmandelbrot(C, max_iter)


# Parameters
width, height = 1000, 800
x_min, x_max, y_min, y_max = -2, 1, -1, 1

# Generate fractal
image = tmandelbrot_set(width, height, x_min, x_max, y_min, y_max).cpu().numpy()

# Display
plt.imshow(image, cmap='inferno', extent=[x_min, x_max, y_min, y_max])
plt.colorbar()
plt.title("Mandelbrot Set")
plt.show()
